In [2]:
!pip install wandb gensim sentence-transformers transformers sentencepiece -q

import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')

import torch
import wandb
wandb.login(key="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")

from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel, T5ForConditionalGeneration

OPTS   = ['A', 'B', 'C', 'D', 'E']
device = "cuda" if torch.cuda.is_available() else "cpu"

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sub   = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("train:", train.shape, "| test:", test.shape, "| device:", device)
train.head(3)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f1000362 (23f1000362-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


train: (2000, 8) | test: (500, 7) | device: cuda


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [3]:
print("=== null counts ===")
print(train.isnull().sum())

print("\n=== answer distribution ===")
print(train['answer'].value_counts())

print("\n=== prompt length (words) ===")
print(train['prompt'].str.split().str.len().describe().round(2))

print("\n=== avg option length (words) ===")
for o in OPTS:
    print(f"  {o}: {train[o].str.split().str.len().mean():.1f}")

=== null counts ===
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

=== answer distribution ===
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

=== prompt length (words) ===
count    2000.00
mean       18.15
std         6.78
min         3.00
25%        14.00
50%        17.00
75%        22.00
max        51.00
Name: prompt, dtype: float64

=== avg option length (words) ===
  A: 26.1
  B: 26.5
  C: 26.5
  D: 26.0
  E: 26.2


In [4]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

def remove_stopwords(tokens):
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS and len(t) > 1]

for col in ['prompt'] + OPTS:
    train[f'clean_{col}'] = train[col].apply(clean_text)
    test[f'clean_{col}']  = test[col].apply(clean_text)

# vocab size of cleaned prompts
all_words = []
for p in train['clean_prompt']:
    all_words.extend(p.split())

print(f"unique vocab in cleaned prompts: {len(set(all_words))}")
print(f"sample — original : {train['prompt'].iloc[0]}")
print(f"sample — cleaned  : {train['clean_prompt'].iloc[0]}")

unique vocab in cleaned prompts: 859
sample — original : Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
sample — cleaned  : pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options


In [5]:
# tfidf fitting on full corpus so test vocab is covered
full_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        full_corpus.append(row['prompt'])
        for o in OPTS:
            full_corpus.append(row[o])

tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=25000, sublinear_tf=True)
tfidf.fit(full_corpus)
print(f"tfidf vocab: {len(tfidf.vocabulary_)}")

# word2vec 
w2v_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        w2v_corpus.append(remove_stopwords(tokenize(row['prompt'])))
        for o in OPTS:
            w2v_corpus.append(remove_stopwords(tokenize(row[o])))

w2v_model = Word2Vec(sentences=w2v_corpus, vector_size=150, window=7, min_count=2, workers=4, sg=1, epochs=15, seed=42)
print(f"w2v vocab: {len(w2v_model.wv)}")

# milestone 2 loading roberta for context aware embeddings
# understands word order and context better 
bert_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
bert_model     = AutoModel.from_pretrained("roberta-base").to(device)
bert_model.eval()  # eval mode so dropout doesnt mess with embeddings

# flan-t5 for zero shot this model can directly answer yes/no if option is correct
t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
t5_model     = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base").to(device)
t5_model.eval()

print("roberta-base and flan-t5-base loaded")

tfidf vocab: 10915
w2v vocab: 2883


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

roberta-base and flan-t5-base loaded


In [7]:
def tfidf_cosine(text1, text2):
    v1 = tfidf.transform([text1])
    v2 = tfidf.transform([text2])
    return float(cosine_similarity(v1, v2)[0][0])

def avg_w2v(text):
    tokens = remove_stopwords(tokenize(text))
    vecs   = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v_model.vector_size)

def w2v_cosine(text1, text2):
    v1 = avg_w2v(text1).reshape(1, -1)
    v2 = avg_w2v(text2).reshape(1, -1)
    if np.all(v1 == 0) or np.all(v2 == 0):
        return 0.0
    return float(cosine_similarity(v1, v2)[0][0])

# milestone 2 roberta embedding function
# taking mean of all token vectors (mean pooling) but ignoring padding tokens using attention mask
def get_roberta_embedding(text):
    inputs = bert_tokenizer(text, return_tensors="pt", truncation=True, max_length=64, padding=True).to(device)
    with torch.no_grad():  # no grad since just doing inference not training
        out = bert_model(**inputs)
    token_embs = out.last_hidden_state
    mask = inputs['attention_mask'].unsqueeze(-1).expand(token_embs.size()).float()
    summed = torch.sum(token_embs * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)  # avoid divide by zero
    return (summed / counts).squeeze(0).cpu().numpy()

# flan-t5 zero shot: asking model directly if the option is correct
# checking logit difference between "yes" and "no" token at first generation step
yes_id = t5_tokenizer("yes", add_special_tokens=False).input_ids[0]
no_id  = t5_tokenizer("no", add_special_tokens=False).input_ids[0]

def flan_score_option(prompt, option_text):
    input_text = f"Question: {prompt}\nAnswer choice: {option_text}\nIs this answer correct? Answer yes or no."
    inputs = t5_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = t5_model.generate(**inputs, max_new_tokens=5, output_scores=True, return_dict_in_generate=True)
    logits = out.scores[0][0]
    return logits[yes_id].item() - logits[no_id].item()

def rank_tfidf(row):
    sims = {o: tfidf_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_w2v(row):
    sims = {o: w2v_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_roberta(row):
    pv = get_roberta_embedding(row['prompt']).reshape(1, -1)
    sims = {o: float(cosine_similarity(pv, get_roberta_embedding(row[o]).reshape(1, -1))[0][0]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_flan_zeroshot(row):
    sims = {o: flan_score_option(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    if actual not in predicted:
        return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted):
        if p == actual:
            hits  += 1
            score += hits / (i + 1.0)
    return score

def mapk(actuals, predictions, k=3):
    return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)

actuals     = train['answer'].tolist()
top3_maj    = list(train['answer'].value_counts().index[:3])
map3_maj    = round(mapk(actuals, [top3_maj] * len(train)), 4)
tfidf_preds = [rank_tfidf(row)[:3] for _, row in train.iterrows()]
map3_tfidf  = round(mapk(actuals, tfidf_preds), 4)
w2v_preds   = [rank_w2v(row)[:3] for _, row in train.iterrows()]
map3_w2v    = round(mapk(actuals, w2v_preds), 4)

# roberta eval on full train
roberta_preds = [rank_roberta(row)[:3] for _, row in train.iterrows()]
map3_roberta  = round(mapk(actuals, roberta_preds), 4)

sample_train = train.sample(n=200, random_state=42)
flan_preds   = [rank_flan_zeroshot(row)[:3] for _, row in sample_train.iterrows()]
map3_flan    = round(mapk(sample_train['answer'].tolist(), flan_preds), 4)

print(f"majority baseline      MAP@3 : {map3_maj}")
print(f"tfidf cosine           MAP@3 : {map3_tfidf}")
print(f"word2vec cosine        MAP@3 : {map3_w2v}")
print(f"roberta cosine         MAP@3 : {map3_roberta}")
print(f"flan-t5 zeroshot (200) MAP@3 : {map3_flan}")

run = wandb.init(
    project = "23f1000362-t22026",
    name    = "milestone2-bert-roberta-flant5",
    config  = {"milestone": 2, "embedding_model": "roberta-base", "zeroshot_model": "flan-t5-base"},
    tags    = ["milestone-2", "transformers"]
)
wandb.log({
    "map3/majority_baseline" : map3_maj,
    "map3/tfidf_cosine"      : map3_tfidf,
    "map3/word2vec_cosine"   : map3_w2v,
    "map3/roberta_cosine"    : map3_roberta,
    "map3/flant5_zeroshot"   : map3_flan
})
wandb.finish()

test_preds = []
for _, row in test.iterrows():
    ranked = rank_flan_zeroshot(row)
    test_preds.append(" ".join(ranked[:3]))

submission = pd.DataFrame({'ID': test['id'].values, 'Prediction': test_preds})
submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv saved")
submission.head(5)

majority baseline      MAP@3 : 0.4213
tfidf cosine           MAP@3 : 0.3228
word2vec cosine        MAP@3 : 0.3028
roberta cosine         MAP@3 : 0.3294
flan-t5 zeroshot (200) MAP@3 : 0.505


map3/flant5_zeroshot,▁
map3/majority_baseline,▁
map3/roberta_cosine,▁
map3/tfidf_cosine,▁
map3/word2vec_cosine,▁
map3/flant5_zeroshot,0.505
map3/majority_baseline,0.4213
map3/roberta_cosine,0.3294
map3/tfidf_cosine,0.3228
map3/word2vec_cosine,0.3028



submission.csv saved


,ID,Prediction
0,1,C D A
1,2,B E D
2,3,D C A
3,4,C E A
4,5,B C D
